# Malicious URL Detection Model Tester

This notebook allows you to test the Random Forest model trained using lexical features as described in the paper:
**"Using Lexical Features for Malicious URL Detection - A Machine Learning Approach"**

In [18]:
import pickle
import pandas as pd
import numpy as np
import tldextract
import re
import os
import requests
import zipfile
import io
from urllib.parse import urlparse

MODEL_PATH = "../models/lexical_rf.pkl"

## 1. Feature Extractor implementation
We use the same 23 lexical features defined in the paper.

In [19]:
class URLFeatureExtractor:
    def __init__(self, top_domains_set=None):
        self.top_domains = top_domains_set if top_domains_set else set()
        self.suspicious_tlds = {'xyz', 'top', 'club', 'win', 'gq', 'cn', 'site', 'online', 'live', 'info', 'loan', 'work'}

    def get_features(self, url):
        if not url.startswith(('http://', 'https://')):
            url = 'http://' + url
            
        parsed = urlparse(url)
        ext = tldextract.extract(url)
        
        primary_domain = ext.domain
        tld = ext.suffix
        subdomain = ext.subdomain
        path = parsed.path
        query = parsed.query
        
        features = {}
        
        # --- 1. URL String Features ---
        features['url_length'] = len(url)
        features['semicolon_count'] = url.count(';')
        features['underscore_count'] = url.count('_')
        features['qmark_count'] = url.count('?')
        features['equal_count'] = url.count('=')
        features['ampersand_count'] = url.count('&')
        
        digits = sum(c.isdigit() for c in url)
        letters = sum(c.isalpha() for c in url)
        features['digit_letter_ratio'] = digits / letters if letters > 0 else 0

        # --- 2. Top Level Domain Features ---
        features['tld_suspicious'] = 1 if tld in self.suspicious_tlds else 0

        # --- 3. Primary Domain Features ---
        ip_pattern = r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}'
        features['domain_has_ip'] = 1 if re.search(ip_pattern, primary_domain) else 0
        features['domain_length'] = len(primary_domain)
        features['domain_digits'] = sum(c.isdigit() for c in primary_domain)
        features['domain_non_alnum'] = sum(not c.isalnum() for c in primary_domain)
        features['domain_hyphens'] = primary_domain.count('-')
        features['domain_at_symbol'] = primary_domain.count('@')
        full_domain = primary_domain + '.' + tld if tld else primary_domain
        features['domain_in_top_list'] = 1 if full_domain in self.top_domains else 0

        # --- 4. Subdomain Features ---
        features['subdomain_dots'] = subdomain.count('.')
        features['subdomain_count'] = len(subdomain.split('.')) if subdomain else 0

        # --- 5. Path Features ---
        features['path_double_slash'] = path.count('//')
        clean_path = path.strip('/')
        features['path_subdirs'] = len(clean_path.split('/')) if clean_path else 0
        features['path_percent20'] = 1 if '%20' in path else 0
        
        path_parts = clean_path.split('/')
        features['path_upper_dirs'] = sum(1 for p in path_parts if any(c.isupper() for c in p))
        features['path_single_char_dirs'] = sum(1 for p in path_parts if len(p) == 1)
        features['path_special_chars'] = sum(not c.isalnum() and c not in ['/', '.'] for c in path)
        features['path_zeroes'] = path.count('0')
        
        path_upper = sum(c.isupper() for c in path)
        path_lower = sum(c.islower() for c in path)
        features['path_upper_lower_ratio'] = path_upper / path_lower if path_lower > 0 else 0

        # --- 6. Query Features ---
        features['query_length'] = len(query)
        features['query_count'] = len(query.split('&')) if query else 0

        return features

## 2. Load Model and Prepare Dependencies

In [20]:
# Load the model
if not os.path.exists(MODEL_PATH):
    print(f"❌ Model file not found at {MODEL_PATH}. Please run the training script first.")
else:
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    print("✅ Model loaded successfully.")

# Optional: Load Tranco Top List to improve feature extraction
top_domains_set = set()
try:
    print("Downloading Tranco top list (this makes 'domain_in_top_list' feature accurate)...")
    tranco_url = "https://tranco-list.eu/top-1m.csv.zip"
    r = requests.get(tranco_url, timeout=10)
    z = zipfile.ZipFile(io.BytesIO(r.content))
    tranco_df = pd.read_csv(z.open('top-1m.csv'), header=None, names=['rank', 'domain'])
    top_domains_set = set(tranco_df['domain'].head(100000).tolist())
    print("✅ Tranco top list loaded (Top 100k).")
except Exception as e:
    print(f"⚠️  Could not load Tranco list: {e}. Model might be less accurate.")

extractor = URLFeatureExtractor(top_domains_set)

✅ Model loaded successfully.
✅ Tranco top list loaded (Top 100k).


## 3. Predict URLs

In [21]:
def predict_url(url):
    """Extracts features and predicts if a URL is malicious."""
    feats = extractor.get_features(url)
    feat_df = pd.DataFrame([feats])
    
    # Get probabilities
    prob = model.predict_proba(feat_df)[0][1]
    prediction = "Malicious" if prob > 0.5 else "Benign"
    
    color = "\033[91m" if prediction == "Malicious" else "\033[92m"
    reset = "\033[0m"
    
    print(f"URL: {url}")
    print(f"Result: {color}{prediction}{reset} (Confidence: {prob:.2%})")
    print("-" * 30)

# Test some samples
predict_url("https://google.com")
predict_url("secure-login-account.top/verify")
predict_url("http://123.45.67.89/malware.exe")

URL: https://google.com
Result: Benign (Confidence: 0.78%)
------------------------------
URL: secure-login-account.top/verify
Result: Malicious (Confidence: 100.00%)
------------------------------
URL: http://123.45.67.89/malware.exe
Result: Malicious (Confidence: 99.00%)
------------------------------


## 4. Test Your Own URL

In [29]:
user_url = "https://gemini.google.com/"  # @param {type:"string"}
if user_url:
    predict_url(user_url)
else:
    print("Enter a URL in the field above (or edit the code) to test it.")

URL: https://gemini.google.com/
Result: Malicious (Confidence: 100.00%)
------------------------------
